**Hasil Analisis Klasifikasi Kualitas Anggur (Wine Quality)**


---

# **1. Persiapan Data**

Pada tahap awal ini, dilakukan pemuatan dataset untuk memahami struktur dan karakteristik data yang akan digunakan dalam pemodelan. Berikut adalah poin-poin penting dari output tersebut:

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Memuat dataset training dan testing
df_train = pd.read_csv('/content/data_training.csv')
df_test = pd.read_csv('/content/data_testing.csv')

# Menampilkan informasi awal data
print("Informasi Dataset Training:")
print(df_train.info())
print("\n5 Data Teratas:")
print(df_train.head())

Informasi Dataset Training:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 857 entries, 0 to 856
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         857 non-null    float64
 1   volatile acidity      857 non-null    float64
 2   citric acid           857 non-null    float64
 3   residual sugar        857 non-null    float64
 4   chlorides             857 non-null    float64
 5   free sulfur dioxide   857 non-null    float64
 6   total sulfur dioxide  857 non-null    float64
 7   density               857 non-null    float64
 8   pH                    857 non-null    float64
 9   sulphates             857 non-null    float64
 10  alcohol               857 non-null    float64
 11  quality               857 non-null    int64  
 12  Id                    857 non-null    int64  
dtypes: float64(11), int64(2)
memory usage: 87.2 KB
None

5 Data Teratas:
   fixed acidity  volatile a

Pada tahap awal ini, dataset pelatihan dimuat sebagai dasar dalam proses pembangunan model klasifikasi untuk memprediksi kualitas anggur. Berdasarkan hasil ringkasan informasi data, dataset tersebut memiliki 857 baris data dengan total 13 kolom. Sebanyak 11 kolom merupakan fitur kimiawi, mulai dari fixed acidity, volatile acidity, hingga alcohol, yang digunakan sebagai variabel independen dalam pemodelan. Sementara itu, variabel yang menjadi target prediksi adalah quality dengan rentang nilai 0–10, sedangkan kolom Id digunakan sebagai penanda unik bagi setiap sampel anggur. Seluruh atribut kimiawi bertipe data float64, sedangkan kolom **quality** dan **Id** bertipe int64. Dari tampilan lima data pertama, terlihat bahwa setiap fitur memiliki rentang nilai yang berbeda cukup jauh, sehingga diperlukan proses normalisasi pada tahap berikutnya agar performa model menjadi lebih optimal dan stabil dalam melakukan prediksi.

# **2. Pembersihan Data** *(Data Cleaning)*

In [2]:
# Cek missing values
print("Missing values pada training set:\n", df_train.isnull().sum())

# Penanganan missing values dengan imputasi median
df_train = df_train.fillna(df_train.median())

# Memisahkan fitur (X) dan target (y)
# Menghapus kolom 'id' karena tidak digunakan untuk melakukan prediksi
X = df_train.drop(columns=['Id', 'quality'])
y = df_train['quality']

# Normalisasi
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

Missing values pada training set:
 fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
Id                      0
dtype: int64


Tahap pembersihan data dilakukan untuk memastikan dataset berada dalam kondisi optimal sebelum digunakan pada proses pelatihan model. Berdasarkan hasil pengecekan missing values, seluruh kolom pada dataset pelatihan tidak memiliki nilai kosong, sehingga tidak diperlukan proses imputasi maupun penghapusan data. Pada tahap ini, fokus utama diarahkan pada transformasi data dengan memisahkan variabel target quality serta menghapus kolom Id karena hanya berfungsi sebagai identitas unik dan tidak memiliki pengaruh terhadap proses prediksi. Setelah itu, diterapkan proses feature scaling menggunakan StandardScaler guna menstandarisasi nilai pada setiap fitur kimiawi. Proses ini penting dilakukan karena terdapat perbedaan rentang nilai yang cukup besar antar variabel, seperti total sulfur dioxide dibandingkan dengan density atau chlorides, sehingga normalisasi diperlukan agar model dapat bekerja lebih stabil dan optimal.

# **3. Pembuatan Model** *(modeling)*

In [3]:
# Membagi data  (80% training, 20% validation)
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Memodelkan dataset menggunakan Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluasi Model
y_pred_val = model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred_val)

print(f"Akurasi Model pada Data Validasi: {accuracy * 100:.2f}%")
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred_val))

Akurasi Model pada Data Validasi: 58.14%

Confusion Matrix:
[[ 0  1  2  0  0]
 [ 0 42 25  0  0]
 [ 0 23 49  6  0]
 [ 0  2 10  9  0]
 [ 0  0  0  3  0]]


Pada tahap pemodelan, digunakan algoritma klasifikasi untuk memprediksi tingkat kualitas anggur berdasarkan karakteristik kimiawi yang tersedia pada dataset pelatihan. Kinerja model kemudian dievaluasi menggunakan data validasi dan menghasilkan nilai akurasi sebesar **58,14%**. Berdasarkan hasil Confusion Matrix, model mampu memberikan prediksi yang relatif baik pada kelas kualitas menengah, khususnya skor 5 dan 6. Namun, model masih mengalami kesulitan dalam mengidentifikasi kelas dengan kualitas sangat rendah maupun sangat tinggi karena distribusi data antar kelas yang tidak seimbang. Penggunaan metrik evaluasi berupa akurasi dan Confusion Matrix bertujuan untuk menilai sejauh mana model mampu melakukan klasifikasi dengan baik sebelum diterapkan pada data pengujian sesungguhnya.

# **4. Prediksi Data Uji** *(Prediction)*

In [4]:
# Persiapan data testing
test_ids = df_test['Id']
X_test = df_test.drop(columns=['Id'])

# Dilakukan scaling yang sama dengan menggunakan data training
X_test_scaled = scaler.transform(X_test)

# Prediksi nilai quality
test_predictions = model.predict(X_test_scaled)

# Menyimpan hasil ke format CSV
submission = pd.DataFrame({
    'Id': test_ids,
    'Quality': test_predictions
})

nama_file = 'hasilprediksi_006.csv'
submission.to_csv(nama_file, index=False)

print(f"Prediksi selesai. File '{nama_file}' telah dibuat.")

Prediksi selesai. File 'hasilprediksi_006.csv' telah dibuat.


Tahap akhir dilakukan dengan menerapkan model yang telah dilatih untuk memprediksi variabel quality pada file **(data_testing.csv)**. Sebelum dilakukan prediksi, data uji terlebih dahulu diproses menggunakan tahapan yang sama seperti data pelatihan, yaitu standarisasi fitur memakai scaler yang telah dibangun sebelumnya agar konsistensi data tetap terjaga. Hasil prediksi kemudian disusun ke dalam file CSV baru yang hanya berisi dua kolom utama, yaitu Id dan nilai prediksi quality, sesuai dengan ketentuan pengerjaan.